In [1]:
!pip install emoji kagglehub

   ---------------------------------------- 0.0/610.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/610.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/610.9 kB ? eta -:--:--
   ---------------------------------- ----- 524.3/610.9 kB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 610.9/610.9 kB 1.7 MB/s  0:00:00


In [10]:
import re
import emoji
from nltk.tokenize import word_tokenize
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE


def clean_tweet(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()  
    text = re.sub(r'\bRT\b', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'#\w+', '', text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['full_text'].apply(clean_tweet)


tokenized_sents = [word_tokenize(text) for text in df['clean_text'].dropna() if len(text) > 0]
train, vocab = padded_everygram_pipeline(2, tokenized_sents)

lm = MLE(2)
lm.fit(train, vocab)


prob = lm.score('is', ['pakistan'])
print(f"Probability of 'is' given 'pakistan': {prob}")

test_data = [('pakistan', 'is')]
perplexity = lm.perplexity(test_data)
print(f"Perplexity of the model on test bigram: {perplexity}")

Probability of 'is' given 'pakistan': 0.05107424114993501
Perplexity of the model on test bigram: 19.579341317365273
